In [1]:
!pip install transformers chromadb sentence-transformers huggingface-hub \
langchain_community langchain-text-splitters pypdf gradio

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 8.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 11.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 14.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 12.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 13.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.7/604.7 kB 15.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 13.6 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -U transformers sentence-transformers chromadb

Defaulting to user installation because normal site-packages is not writeable


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("totalenergies_sustainability-climate-2025-progress-report_2025_en.pdf")
docs = loader.load()

print("Number of pages:", len(docs))

/Users/augustin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/augustin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of pages: 120


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(docs)

text_lines = [chunk.page_content for chunk in chunks]

print("Number of chunks:", len(text_lines))

Number of chunks: 476


In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def emb_text(text):
    return embedding_model.encode(
        [text],
        normalize_embeddings=True
    ).tolist()[0]

In [6]:
from chromadb import PersistentClient

client = PersistentClient(path="./rag_db")

collection_name = "esg_rag"

try:
    collection = client.get_collection(name=collection_name)
except:
    collection = client.create_collection(name=collection_name)

In [7]:
from tqdm import tqdm

embeddings = []
ids = []

for i, line in enumerate(tqdm(text_lines)):
    embeddings.append(emb_text(line))
    ids.append(str(i))

collection.add(
    documents=text_lines,
    embeddings=embeddings,
    ids=ids
)

100%|██████████| 476/476 [00:12<00:00, 39.24it/s]


In [8]:
# V1
#def retrieve_context(question, n_results=2):

    #query_embedding = emb_text(question)

    #results = collection.query(
        #query_embeddings=[query_embedding],
        #n_results=n_results
    #)

    #context = "\n".join(results["documents"][0])

    #return context

In [9]:
# V2
#def retrieve_context(question, n_results=8):

    #query_embedding = emb_text(question)

    #results = collection.query(
        #query_embeddings=[query_embedding],
        #n_results=n_results
    #)

    #context = ""
    #for chunk in results["documents"][0]:
        #if len(context) + len(chunk) < 1500:
            #context += chunk + "\n"

    #return context

In [10]:
# V3
def retrieve_context(question, n_results=8):

    query_embedding = emb_text(question)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    docs = results["documents"][0]

    keywords = ["climate", "emissions", "carbon", "strategy", "target"]

    filtered_docs = []
    for doc in docs:
        if any(k in doc.lower() for k in keywords):
            filtered_docs.append(doc)

    if len(filtered_docs) == 0:
        filtered_docs = docs[:3]

    context = ""
    for i, chunk in enumerate(filtered_docs):
      if len(context) + len(chunk) < 1500:
        context += f"[Source {i+1}]\n{chunk}\n\n"

    return context

In [11]:
# V1
PROMPT = """
You are an expert analyst.

Using ONLY the context below, answer the question.

IMPORTANT:
- Combine ALL relevant information from the context
- List the main points clearly
- Be specific and complete

Context:
{context}

Question:
{question}

Answer (structured):
"""

In [12]:
# V2
PROMPT = """
You are an expert assistant.

Answer the question using ALL relevant information from the context.
Be clear and detailed.

Context:
{context}

Question:
{question}

Answer:
"""

In [13]:
# V3
PROMPT = """
Answer the question using ALL relevant information from the context.

Context:
{context}

Question:
{question}

Answer:
"""

In [14]:
# V4 (To prevent hallucinations)
PROMPT = """
You are an expert analyst.

Answer the question using ONLY the information explicitly present in the context.

IMPORTANT:
- Do NOT add any information that is not in the context
- Do NOT invent or assume anything
- If information is missing, do not guess
- Provide a clear and structured answer

Context:
{context}

Question:
{question}

Answer:
"""

In [15]:
# V5 (Best so far)
PROMPT = """
Answer the question using ONLY the context.

IMPORTANT:
- Do NOT add information not present in the context
- Be concise and factual
- Provide 3 to 5 bullet points maximum

Context:
{context}

Question:
{question}

Answer:
"""

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

#model_name = "google/flan-t5-base"
model_name = "Qwen/Qwen2.5-0.5B"
#model_name = "Qwen/Qwen2.5-1.5B"
#model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForSeq2SeqLM.from_pretrained(model_name)    for "google/flan-t5-base"
model = AutoModelForCausalLM.from_pretrained(model_name)

In [17]:
# V1 (1st model)
#def generate_answer(prompt):

    #inputs = tokenizer(
        #prompt,
        #return_tensors="pt",
        #truncation=True,
        #max_length=512
    #)

    #outputs = model.generate(
        #**inputs,
        #max_new_tokens=200,
        #temperature=0.3,
        #do_sample=4
    #)

    #return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [18]:
# V2 (the prompt is returned in the answer)
#def generate_answer(prompt):

    #inputs = tokenizer(
        #prompt,
        #return_tensors="pt",
        #truncation=True,
        #max_length=512
    #)

    #outputs = model.generate(
        #**inputs,
        #max_new_tokens=200,
        #temperature=0.7,
        #do_sample=True
    #)

    #return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [19]:
def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        temperature=0.2,
        do_sample=True
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = full_output[len(prompt):]

    return clean_answer(answer)

In [20]:
def clean_answer(answer):

    lines = answer.split("\n")

    cleaned = []
    for line in lines:
        if len(line.strip()) > 0:
            cleaned.append(line.strip())

    return "\n".join(cleaned)

In [21]:
def ask_rag(question):

    context = retrieve_context(question)

    prompt = PROMPT.format(
        context=context,
        question=question
    )

    return generate_answer(prompt)

In [22]:
ask_rag("What are TotalEnergies' main climate goals and targets?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


"TotalEnergies' main climate goals and targets are to reduce its net Scope 1+2 emissions by 40% by 2030 relative to 2015, after mobilizing around 5 million credits from nature-based carbon sinks projects."

In [23]:
ask_rag("What is the company's goal for carbon neutrality?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


"The company's goal for carbon neutrality is to eliminate the equivalent of 100 Mt/year of CO2 generated by its customers by developing carbon utilization (CCU) and carbon capture and storage (CCS) solutions."

In [24]:
ask_rag("What principles guide TotalEnergies' climate commitments?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


"TotalEnergies' climate commitments are guided by the six principles outlined in their public and responsible commitment to climate change within industry associations. These principles include:\n1. **TotalEnergies recognizes the link established by science between human activities, in particular the use of fossil fuels, and climate change.**\n2. **TotalEnergies recognizes the Paris Agreement as a major step forward in the fight against global warming and supports the initiatives of the implementing States to fulfill its aims.**\n3. **TotalEnergies supports the implementation of carbon pricing mechanisms.**"

In [ ]:
import gradio as gr

def rag_interface(question, history):
    answer = ask_rag(question)
    return answer

demo = gr.ChatInterface(
    fn=rag_interface,
    title="TotalEnergies ESG Assistant 🌿",
    description="Ask any question about the TotalEnergies Sustainability Report 2025. Powered by ChromaDB + Qwen2.5-0.5B.",
    examples=[
        "What are TotalEnergies' main climate goals and targets?",
        "What is the company's goal for carbon neutrality?",
        "What principles guide TotalEnergies' climate commitments?",
    ],
    theme=gr.themes.Soft(),
)

demo.launch(share=True)